# Main revision experiments — organized execution copy

This copy is derived from the executed notebook archived at `archive/revision_executed/01_Revision_Main_Experiments.ipynb`. Historical outputs remain only in that original and the saved result CSVs. Empty outputs here do not indicate a new experiment.

**Not yet GPU-tested as a clean session.** Install dependencies, restart the session, then continue with paths. Required audio/checkpoints and historical outputs for the reproduction check/E7 are not included in the repository. Consult `docs/RUN_ORDER.md`. Do not run all cells without preparing those prerequisites. The install pins are historical and still need environment validation.

The inference functions, model parameters, seeds, repetitions and scoring code are unchanged. Changes: fixed E9 columns, strict atomic result writing, removed historical recovery/diagnostic cells from this execution copy. Existing malformed CSV files cause a stop; use the archived recovery code separately after making a backup.

`OUT` still points to the historical `revision_runs_final` directory. Keep the published results immutable: use a separate research-data directory for a reproduction, update BASE_DIR consistently in all related notebooks, and retain the original source/target files and metadata selection. This copy does not create missing data automatically.


In [ ]:
# Cell 1 — INSTALL
# After completion: Runtime → Restart session

import os
import sys
import subprocess

print("Python:", sys.version, flush=True)

OPENVOICE_REPO = "/content/OpenVoice"
OPENVOICE_COMMIT = "74a1d147b17a8c3092dd5430504bd83ef6c7eb23"

SEED_REPO = "/content/seed-vc"
SEED_COMMIT = "51383efd921027683c89e5348211d93ff12ac2a8"

repositories = [
    (
        OPENVOICE_REPO,
        "https://github.com/myshell-ai/OpenVoice.git",
        OPENVOICE_COMMIT,
    ),
    (
        SEED_REPO,
        "https://github.com/Plachtaa/seed-vc.git",
        SEED_COMMIT,
    ),
]

for repo, url, commit in repositories:
    print(f"\nPreparing {repo}...", flush=True)

    if not os.path.isdir(repo):
        subprocess.run(
            ["git", "clone", "--progress", url, repo],
            check=True,
        )

    subprocess.run(
        ["git", "checkout", commit],
        cwd=repo,
        check=True,
    )

    print(f"Repository ready: {repo}", flush=True)

pkgs = [
    # OpenVoice
    "wavmark==0.0.3",
    "eng_to_ipa==0.0.2",
    "inflect==7.0.0",
    "unidecode==1.3.7",
    "pypinyin==0.50.0",
    "cn2an==0.5.22",
    "jieba==0.42.1",
    "langid==1.1.6",

    # Seed-VC
    "numpy==1.26.4",
    "scipy==1.13.1",
    "librosa==0.10.2",
    "transformers==4.46.3",
    "huggingface-hub>=0.28.1",
    "munch==4.0.0",
    "einops==0.8.0",
    "descript-audio-codec==1.0.0",
    "resemblyzer",
    "soundfile==0.12.1",
    "modelscope==1.18.1",
    "funasr==1.1.5",
    "hydra-core==1.3.2",
    "python-dotenv",
    "pyyaml",

    # Scoring
    "speechbrain==1.0.3",
    "jiwer==3.0.3",
    "accelerate",
]

print("\nInstalling packages...", flush=True)

process = subprocess.Popen(
    [
        sys.executable,
        "-u",
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "off",
    ] + pkgs,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Package installation failed: {return_code}"
    )

print(
    "\n✅ Installed. NOW: Runtime → Restart session, "
    "then run Cell 2.",
    flush=True,
)

In [ ]:
# Runtime configuration — edit these two lines consistently in all notebooks.
import os
DATA_ROOT = "/content/drive/MyDrive/IJEEE_VC_data"
RUN_DIR = "/content/drive/MyDrive/IJEEE_VC_data/reproduction_runs/review_run_01"
# DATA_ROOT contains audio/models. RUN_DIR is a new output folder, never results/.
os.environ["IJEEE_DATA_ROOT"] = DATA_ROOT
os.environ["IJEEE_RUN_DIR"] = RUN_DIR


In [ ]:
# Cell 2 — PATHS AND SETTINGS
import os, sys, glob, json, time, random, shutil, tempfile, re, string, unicodedata, subprocess
os.environ["USE_TF"] = "0"; os.environ["TRANSFORMERS_NO_TF"] = "1"; os.environ["USE_FLAX"] = "0"
from google.colab import drive
drive.mount("/content/drive")
import numpy as np, pandas as pd, torch

BASE_DIR = os.environ.get("IJEEE_DATA_ROOT", "/content/drive/MyDrive/IJEEE_VC_data")
OUT = os.environ.get("IJEEE_RUN_DIR", os.path.join(BASE_DIR, "reproduction_runs", "review_run_01")); os.makedirs(OUT, exist_ok=True)

TUNING_FILES = [os.path.join(BASE_DIR, "selected_hybrid_data", f"joe_source_{i:02d}.wav") for i in (1, 2, 3)]
TUNING_TEXT = ["The monkey was planning to throw the net like the fishermen did.",
               "Gaze out to sea, picnic on benches and watch fishermen hook herring.",
               "We all have our own methods of dealing with money, but here are a couple of saving tricks that will grow your money."]
TARGET_REF = os.path.join(BASE_DIR, "selected_hybrid_data", "kathleen_reference.wav")
# EDIT if the two new conditioning references are stored elsewhere on Drive:
EXTRA_REFS = [os.path.join(BASE_DIR, "selected_hybrid_data", f"kathleen_conditioning_ref_{i:02d}.wav") for i in (2, 3)]
EVAL_CLIPS = sorted(glob.glob(os.path.join(BASE_DIR, "kathleen_independent_refs", "kathleen_eval_*.wav")))
JOE_ARCHIVE = os.path.join(BASE_DIR, "joe.tar.gz")
HOLDOUT10_META = os.path.join(BASE_DIR, "final_holdout_data", "holdout_metadata.csv")
HOLDOUT40_DIR = os.path.join(BASE_DIR, "final_holdout_data_40"); os.makedirs(HOLDOUT40_DIR, exist_ok=True)
HOLDOUT40_META = os.path.join(HOLDOUT40_DIR, "holdout40_metadata.csv")
EXISTING = os.path.join(BASE_DIR, "final_hybrid_comparison")
OV_CONVERTER = os.path.join(BASE_DIR, "models", "OpenVoiceV2", "converter")
N_HOLDOUT = 40
TUNING_IDS = {"0000000160", "0000000229", "0000000068"}

for p in TUNING_FILES + [TARGET_REF, JOE_ARCHIVE, HOLDOUT10_META, os.path.join(OV_CONVERTER, "checkpoint.pth")]:
    assert os.path.isfile(p), f"❌ missing: {p}"
for p in EXTRA_REFS:
    print("extra reference", os.path.basename(p), "found" if os.path.isfile(p) else "❌ NOT FOUND — fix the EDIT line")
print("Independent Kathleen evaluation clips:", len(EVAL_CLIPS))
assert len(EVAL_CLIPS) == 10, "❌ kathleen_eval_*.wav clips not found"
assert torch.cuda.is_available(), "❌ select a T4 GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))

def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def append_row(csv, row):
    # Schema validation and atomic replacement happen outside the measured span.
    # Existing malformed files are rejected, never silently repaired or skipped.
    import csv as csv_module
    import io
    import tempfile
    target = os.path.abspath(csv)
    parent = os.path.dirname(target)
    incoming = list(row)
    old_rows = []
    if os.path.exists(target) and os.path.getsize(target):
        with open(target, newline="", encoding="utf-8") as handle:
            reader = csv_module.reader(handle, strict=True)
            columns = next(reader)
            if len(columns) != len(set(columns)):
                raise ValueError(f"Duplicate CSV columns: {target}")
            if set(columns) != set(incoming):
                raise ValueError(f"CSV schema differs: {target}; preserve the file and inspect it before resuming")
            for number, values in enumerate(reader, start=2):
                if len(values) != len(columns):
                    raise ValueError(f"Malformed CSV row {number}: {target}; historical recovery must be performed separately")
                old_rows.append(values)
    else:
        columns = incoming
    # pandas encodes floats, NaN and transcript quoting as in the original code.
    encoded = pd.DataFrame([row], columns=columns).to_csv(index=False, header=False)
    new_values = list(csv_module.reader(io.StringIO(encoded), strict=True))
    if len(new_values) != 1 or len(new_values[0]) != len(columns):
        raise ValueError("Expected exactly one valid result row")
    descriptor, temporary = tempfile.mkstemp(prefix=".result_", suffix=".csv.tmp", dir=parent)
    try:
        with os.fdopen(descriptor, "w", newline="", encoding="utf-8") as handle:
            writer = csv_module.writer(handle)
            writer.writerow(columns)
            writer.writerows(old_rows)
            writer.writerow(new_values[0])
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, target)
    finally:
        if os.path.exists(temporary):
            os.remove(temporary)
def done_keys(csv, cols):
    if not os.path.exists(csv): return set()
    return set(map(tuple, pd.read_csv(csv)[cols].astype(str).values))
print("✅ Paths ready")

In [ ]:
# Cell 3 — EXTEND THE HOLDOUT SET TO 40 SENTENCES

from pathlib import Path
import tarfile

LOCAL_JOE = "/content/final_comparison_dataset/joe"
os.makedirs(LOCAL_JOE, exist_ok=True)

if not glob.glob(
    os.path.join(LOCAL_JOE, "**", "*.webm"),
    recursive=True,
):
    with tarfile.open(JOE_ARCHIVE, "r:gz") as archive:
        archive.extractall(LOCAL_JOE, filter="data")

files = []

for extension in [
    "*.webm", "*.wav", "*.flac", "*.mp3",
    "*.ogg", "*.opus", "*.m4a",
]:
    files.extend(
        glob.glob(
            os.path.join(LOCAL_JOE, "**", extension),
            recursive=True,
        )
    )

files = sorted(set(files))
print("Joe files:", len(files))

candidates = [
    path
    for path in files
    if Path(path).stem not in TUNING_IDS
]

random.Random(20260822).shuffle(candidates)


def dur(path):
    result = subprocess.run(
        [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            path,
        ],
        capture_output=True,
        text=True,
    )

    try:
        return float(result.stdout.strip())
    except ValueError:
        return None


def txt(path):
    transcript_path = Path(path).with_suffix(".txt")

    if not transcript_path.is_file():
        return None

    transcript = " ".join(
        transcript_path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).split()
    )

    return transcript or None


records = []

for path in candidates:
    if len(records) == N_HOLDOUT:
        break

    duration = dur(path)

    if duration is None or not (4.0 < duration < 7.5):
        continue

    transcript = txt(path)

    if transcript is None:
        continue

    sample_number = len(records) + 1

    output_path = os.path.join(
        HOLDOUT40_DIR,
        f"joe_holdout_{sample_number:02d}.wav",
    )

    if not os.path.isfile(output_path):
        subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-v", "error",
                "-i", path,
                "-ac", "1",
                "-ar", "22050",
                "-c:a", "pcm_s16le",
                output_path,
            ],
            check=True,
        )

    records.append(
        {
            "sample": sample_number,
            "original_id": Path(path).stem,
            "duration_seconds": duration,
            "transcript": transcript,
            "prepared_wav": output_path,
        }
    )

    print(
        f"Prepared {sample_number}/{N_HOLDOUT}: "
        f"{Path(path).stem}",
        flush=True,
    )

assert len(records) == N_HOLDOUT, (
    f"❌ Expected {N_HOLDOUT} sentences, found {len(records)}"
)

H40 = pd.DataFrame(records)

# Preserve leading zeros in recording IDs.
old10 = pd.read_csv(
    HOLDOUT10_META,
    dtype={"original_id": str},
)

same = (
    H40["original_id"].astype(str).head(10).tolist()
    == old10["original_id"].astype(str).tolist()
)

print("First 10 identical to the paper's holdout:", same)

assert same, (
    "❌ The extended selection does not start with "
    "the original 10 — stop and report"
)

H40.to_csv(HOLDOUT40_META, index=False)

print(
    H40[
        ["sample", "original_id", "duration_seconds"]
    ].tail(5)
)

print("✅ 40 holdout sentences ready")

In [ ]:
# Cell 4 — LOAD MODELS, WARM-UP, CONVERSION FUNCTIONS (from Untitled15)
from types import SimpleNamespace
sys.path.insert(0, "/content/OpenVoice")
from openvoice.api import ToneColorConverter
tone_color_converter = ToneColorConverter(os.path.join(OV_CONVERTER, "config.json"), device="cuda:0")
tone_color_converter.load_ckpt(os.path.join(OV_CONVERTER, "checkpoint.pth"))
sys.path.insert(0, "/content/seed-vc"); os.chdir("/content/seed-vc")
import inference as seed_inference
def seed_args(src, ref, out, steps):
    return SimpleNamespace(source=src, target=ref, output=out, diffusion_steps=steps, length_adjust=1.0,
                           inference_cfg_rate=0.7, f0_condition=False, auto_f0_adjust=False, semi_tone_shift=0,
                           checkpoint=None, config=None, fp16=True)
seed_model_bundle = seed_inference.load_models(seed_args(None, None, None, 10))
seed_inference.load_models = lambda args: seed_model_bundle       # never reload inside main()

SE_CACHE = {}
def get_se(path):                     # cached, outside the timed span (as in the paper)
    if path not in SE_CACHE: SE_CACHE[path] = tone_color_converter.extract_se(path)
    return SE_CACHE[path]
def run_openvoice(src, ref, tau, out_path, seed):
    src_se, tgt_se = get_se(src), get_se(ref)
    set_all_seeds(seed); torch.cuda.synchronize(); t0 = time.perf_counter()
    tone_color_converter.convert(audio_src_path=src, src_se=src_se, tgt_se=tgt_se, output_path=out_path, tau=tau, message="@MyShell")
    torch.cuda.synchronize(); return time.perf_counter() - t0
os.makedirs("/content/seed_tmp", exist_ok=True)
def run_seedvc(src, ref, steps, out_path, seed):
    work = tempfile.mkdtemp(dir="/content/seed_tmp")
    set_all_seeds(seed); torch.cuda.synchronize(); t0 = time.perf_counter()
    seed_inference.main(seed_args(src, ref, work, steps))
    torch.cuda.synchronize(); el = time.perf_counter() - t0
    produced = sorted(glob.glob(os.path.join(work, "*.wav")), key=os.path.getmtime)
    assert produced, "❌ Seed-VC produced no file"
    shutil.move(produced[-1], out_path); shutil.rmtree(work, ignore_errors=True)
    return el
def run_cascade_e2e(src, ref, tau, steps, out_path, ov_seed, sv_seed):
    # one end-to-end timed run: both stages and the intermediate file inside one timed span
    inter = out_path.replace(".wav", "_e2e_ov.wav")
    torch.cuda.synchronize(); t0 = time.perf_counter()
    run_openvoice(src, ref, tau, inter, ov_seed); run_seedvc(inter, ref, steps, out_path, sv_seed)
    torch.cuda.synchronize(); return time.perf_counter() - t0

WARM = "/content/warmup"; os.makedirs(WARM, exist_ok=True)
for _ in range(2):
    run_openvoice(TUNING_FILES[0], TARGET_REF, 0.5, f"{WARM}/ov.wav", 2026)
    run_seedvc(TUNING_FILES[0], TARGET_REF, 10, f"{WARM}/sv.wav", 2028)
ENV = dict(gpu=torch.cuda.get_device_name(0), torch=torch.__version__, openvoice_commit="74a1d147b17a8c3092dd5430504bd83ef6c7eb23",
           seedvc_commit="51383efd921027683c89e5348211d93ff12ac2a8", seedvc_fp16=True, openvoice_precision="fp32",
           batch_size=1, warmups=2, timed_repeats=3, cuda_synchronize=True, openvoice_embeddings_cached=True)
json.dump(ENV, open(os.path.join(OUT, "environment.json"), "w"), indent=1)
print("✅ Models loaded and warmed up"); print(ENV)

In [ ]:
# Cell 5 — SCORING MODELS (ECAPA and Whisper-small exactly as in Untitled13; plus WavLM, Whisper-large-v3, UTMOS)
import librosa, jiwer, torchaudio, logging
import torch.nn.functional as F
if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["soundfile"]
from speechbrain.inference.speaker import SpeakerRecognition
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, AutoFeatureExtractor, WavLMForXVector
logging.getLogger("speechbrain").setLevel(logging.WARNING)
DEV = "cuda"
speaker_model = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb",
                                                savedir="/content/evaluation_models/ecapa_voxceleb", run_opts={"device": DEV})
ASR = {}
for name, mid, dtype in [("small", "openai/whisper-small", torch.float32), ("large", "openai/whisper-large-v3", torch.float16)]:
    proc = AutoProcessor.from_pretrained(mid)
    mdl = AutoModelForSpeechSeq2Seq.from_pretrained(mid, torch_dtype=dtype, use_safetensors=True).to(DEV).eval()
    mdl.config.forced_decoder_ids = None; mdl.generation_config.forced_decoder_ids = None
    ASR[name] = (proc, mdl, dtype)
wl_fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
wl = WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv").to(DEV).eval()
utmos = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).to(DEV).eval()

def load_audio_16k(p): return np.asarray(librosa.load(p, sr=16000, mono=True)[0], dtype=np.float32)
_EMB = {}
@torch.inference_mode()
def emb(p, kind):
    if (p, kind) in _EMB: return _EMB[(p, kind)]
    w = load_audio_16k(p)
    if kind == "ecapa":   # identical to Untitled13 extract_speaker_embedding
        e = speaker_model.encode_batch(torch.from_numpy(w).unsqueeze(0).to(DEV), wav_lens=torch.tensor([1.0]).to(DEV), normalize=False)
        e = F.normalize(e.squeeze().float(), p=2, dim=0).cpu()
    else:
        x = wl_fe(w, sampling_rate=16000, return_tensors="pt").to(DEV)
        e = F.normalize(wl(**x).embeddings.squeeze().float(), p=2, dim=0).cpu()
    _EMB[(p, kind)] = e
    return e
def cos(a, b, kind): return float(F.cosine_similarity(emb(a, kind), emb(b, kind), dim=0))

def join_spelled_letters(t):          # "N E U" -> "NEU" (reviewer fix), applied to reference and hypothesis
    return re.sub(r"\b(?:[A-Z] )+[A-Z]\b", lambda m: m.group(0).replace(" ", ""), str(t))
def normalize_transcript(text):        # identical to Untitled13 after the letter-joining step
    text = unicodedata.normalize("NFKC", join_spelled_letters(text)).lower()
    for a, b in [("’", "'"), ("–", " "), ("—", " "), ("-", " ")]: text = text.replace(a, b)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()
@torch.inference_mode()
def transcribe(p, which):
    proc, mdl, dtype = ASR[which]
    mi = proc(load_audio_16k(p), sampling_rate=16000, return_tensors="pt", return_attention_mask=True)
    args = dict(input_features=mi.input_features.to(DEV, dtype), language="en", task="transcribe", max_new_tokens=128)
    if getattr(mi, "attention_mask", None) is not None: args["attention_mask"] = mi.attention_mask.to(DEV)
    return proc.batch_decode(mdl.generate(**args), skip_special_tokens=True)[0].strip()
@torch.inference_mode()
def mos(p): return float(utmos(torch.from_numpy(load_audio_16k(p)).unsqueeze(0).to(DEV), 16000).item())

def score(out, text, src, cond_ref, eval_clips):
    r = dict(sim_ecapa_cond=cos(out, cond_ref, "ecapa"), sim_wavlm_cond=cos(out, cond_ref, "wavlm"),
             sim_ecapa_eval=float(np.mean([cos(out, c, "ecapa") for c in eval_clips])),
             sim_wavlm_eval=float(np.mean([cos(out, c, "wavlm") for c in eval_clips])),
             leak_ecapa_source=cos(out, src, "ecapa"), utmos=mos(out), duration_out=librosa.get_duration(path=out))
    ref_n = normalize_transcript(text)
    for which in ("small", "large"):
        hyp = transcribe(out, which); hyp_n = normalize_transcript(hyp)
        r[f"hyp_{which}"] = hyp; r[f"wer_{which}"] = jiwer.wer(ref_n, hyp_n); r[f"cer_{which}"] = jiwer.cer(ref_n, hyp_n)
        r[f"ref_words"] = len(ref_n.split())
    for k in list(_EMB):                 # keep only cached reference/eval/source embeddings
        if k[0] == out: _EMB.pop(k)
    return r
print("✅ Scoring models ready")

In [ ]:
# Cell 6 — REPRODUCTION CHECK (must pass)
H40 = pd.read_csv(HOLDOUT40_META)
f = os.path.join(EXISTING, "seed_based_outputs", "optimized_hybrid", "optimized_hybrid_sample_01_tau_0.5_steps_10.wav")
s = score(f, H40.transcript[0], H40.prepared_wav[0], TARGET_REF, EVAL_CLIPS)
print(f"Existing output: similarity {s['sim_ecapa_cond']:.4f} (paper CSV 0.6885) | WER small {s['wer_small']:.4f} (paper 0.1667)")
assert abs(s["sim_ecapa_cond"] - 0.6885) < 0.01, "❌ scoring differs from the paper — stop and report"
new = os.path.join(OUT, "_check_selected_s01.wav")
run_openvoice(H40.prepared_wav[0], TARGET_REF, 0.5, new.replace(".wav", "_ov.wav"), 202701)
run_seedvc(new.replace(".wav", "_ov.wav"), TARGET_REF, 10, new, 202901)
t = score(new, H40.transcript[0], H40.prepared_wav[0], TARGET_REF, EVAL_CLIPS)
print(f"Regenerated with the paper's seeds: similarity {t['sim_ecapa_cond']:.4f} | transcript: {t['hyp_small']}")
print("✅ REPRODUCTION CHECK PASSED" if abs(t["sim_ecapa_cond"] - 0.6885) < 0.03 else "⚠️ regenerated output differs — report before continuing")

In [ ]:
# Cell 7 — RESCORE THE 40 EXISTING OUTPUTS + ANCHORS (no conversion)
csv = os.path.join(OUT, "E7_rescore_existing.csv"); done = done_keys(csv, ["system", "sample"])
pat = {"OpenVoice V2": "openvoice_outputs/tau_0.5/openvoice_sample_{:02d}_tau_0.5.wav",
       "Seed-VC": "seed_based_outputs/seed_only/seed_only_sample_{:02d}_steps_10.wav",
       "Reference cascade": "seed_based_outputs/default_hybrid/default_hybrid_sample_{:02d}_tau_0.3_steps_20.wav",
       "Selected cascade": "seed_based_outputs/optimized_hybrid/optimized_hybrid_sample_{:02d}_tau_0.5_steps_10.wav"}
for sysname, pattern in pat.items():
    for i in range(1, 11):
        if (sysname, str(i)) in done: continue
        r = H40.iloc[i - 1]
        append_row(csv, dict(system=sysname, sample=i, **score(os.path.join(EXISTING, pattern.format(i)), r.transcript, r.prepared_wav, TARGET_REF, EVAL_CLIPS)))
acsv = os.path.join(OUT, "E7_anchors.csv")
if not os.path.exists(acsv):
    for c in EVAL_CLIPS:
        append_row(acsv, dict(anchor="target-target", file=os.path.basename(c), sim_ecapa=cos(c, TARGET_REF, "ecapa"),
                              sim_wavlm=cos(c, TARGET_REF, "wavlm"), utmos=mos(c)))
    for _, r in H40.iterrows():
        append_row(acsv, dict(anchor="source-target", file=os.path.basename(r.prepared_wav), sim_ecapa=cos(r.prepared_wav, TARGET_REF, "ecapa"),
                              sim_wavlm=cos(r.prepared_wav, TARGET_REF, "wavlm"), utmos=mos(r.prepared_wav)))
a = pd.read_csv(acsv)
print(pd.read_csv(csv).groupby("system").mean(numeric_only=True).round(4).T)
print(a.groupby("anchor").mean(numeric_only=True).round(4))
print("Eval clips nearly identical to the conditioning reference (ECAPA > 0.95):", int((a[a.anchor == "target-target"].sim_ecapa > 0.95).sum()))

In [ ]:
# Cell 8 — FULL 5 x 5 GRID ON THE 3 TUNING SENTENCES, 5 SEEDS (OpenVoice output reused across step counts, as in Table III)
csv = os.path.join(OUT, "E8_grid_5x5_seeds.csv"); done = done_keys(csv, ["tau", "steps", "sample", "seed_set"])
TAUS, STEPS = [0.1, 0.3, 0.5, 0.7, 0.9], [5, 10, 20, 30, 50]
d8 = os.path.join(OUT, "E8_audio"); os.makedirs(d8, exist_ok=True)
for k in range(1, 6):
    for tau in TAUS:
        for i, (src, text) in enumerate(zip(TUNING_FILES, TUNING_TEXT), 1):
            ov_seed, sv_seed = 400000 + 1000 * k + i, 410000 + 1000 * k + i
            if all((str(tau), str(st), str(i), str(k)) in done for st in STEPS): continue
            ov_out = os.path.join(d8, f"ov_tau{tau}_s{i}_k{k}.wav")
            t_ov = run_openvoice(src, TARGET_REF, tau, ov_out, ov_seed)
            for st in STEPS:
                if (str(tau), str(st), str(i), str(k)) in done: continue
                out = os.path.join(d8, f"casc_tau{tau}_st{st}_s{i}_k{k}.wav")
                t_sv = run_seedvc(ov_out, TARGET_REF, st, out, sv_seed)
                append_row(csv, dict(tau=tau, steps=st, sample=i, seed_set=k, ov_seed=ov_seed, sv_seed=sv_seed,
                                     t_openvoice=t_ov, t_seedvc=t_sv, latency=t_ov + t_sv, src_duration=librosa.get_duration(path=src),
                                     **score(out, text, src, TARGET_REF, EVAL_CLIPS)))
g = pd.read_csv(csv)
print(g.pivot_table(index="tau", columns="steps", values="sim_ecapa_cond").round(4))
print(g.pivot_table(index="tau", columns="steps", values="wer_large").round(4))

In [ ]:
g = pd.read_csv(os.path.join(OUT, "E8_grid_5x5_seeds.csv"))
counts = g.drop_duplicates(
    ["tau", "steps", "sample", "seed_set"]
).groupby(["tau", "steps"]).size()

print("عدد النتائج:", counts.sum(), "من 375")
print("كل تركيبة فيها 15 نتيجة:", len(counts) == 25 and counts.eq(15).all())

In [ ]:
# Cell 9 — HOLDOUT: 40 sentences x 7 systems; paper timing (3 runs per stage) + one end-to-end cascade run
csv = os.path.join(OUT, "E9_holdout40.csv"); done = done_keys(csv, ["system", "sample"])
d9 = os.path.join(OUT, "E9_audio"); os.makedirs(d9, exist_ok=True)
SYSTEMS = [("OpenVoice V2 (tau 0.5)", "ov", 0.5, None), ("Seed-VC (10 steps)", "sv", None, 10),
           ("Seed-VC (25 steps, default)", "sv", None, 25), ("Reference cascade (tau 0.3, 20)", "casc", 0.3, 20),
           ("Selected cascade (tau 0.5, 10)", "casc", 0.5, 10), ("Cascade (tau 0.3, 10)", "casc", 0.3, 10),
           ("Cascade (tau 0.5, 20)", "casc", 0.5, 20)]
OV_T = {}                                   # mean OpenVoice time per (tau, sample), reused by cascades as in the paper
def ov_stage(i, src, tau):
    key = (tau, i); out = os.path.join(d9, f"ov_tau{tau}_s{i:02d}.wav")
    if key not in OV_T:
        OV_T[key] = (float(np.mean([run_openvoice(src, TARGET_REF, tau, out, 202700 + i) for _ in range(3)])), out)
    return OV_T[key]
for name, kind, tau, steps in SYSTEMS:
    for _, r in H40.iterrows():
        i = int(r["sample"])
        if (name, str(i)) in done: continue
        src, tag = r.prepared_wav, re.sub(r"[^A-Za-z0-9]+", "_", name)
        out = os.path.join(d9, f"{tag}_s{i:02d}.wav"); row = dict(system=name, sample=i, tau=tau, steps=steps, src_duration=r.duration_seconds, latency_e2e=np.nan)
        if kind == "ov":
            t, ov_out = ov_stage(i, src, tau); shutil.copy2(ov_out, out); row.update(t_openvoice=t, t_seedvc=0.0, latency=t)
        else:
            if kind == "sv": t_ov, inp = 0.0, src
            else: t_ov, inp = ov_stage(i, src, tau)
            t_sv = float(np.mean([run_seedvc(inp, TARGET_REF, steps, out, 202900 + i) for _ in range(3)]))
            row.update(t_openvoice=t_ov, t_seedvc=t_sv, latency=t_ov + t_sv)
            if kind == "casc":
                row["latency_e2e"] = run_cascade_e2e(src, TARGET_REF, tau, steps, out.replace(".wav", "_e2e.wav"), 202700 + i, 202900 + i)
        row["rtf"] = row["latency"] / r.duration_seconds
        append_row(csv, dict(**row, **score(out, r.transcript, src, TARGET_REF, EVAL_CLIPS)))
h = pd.read_csv(csv)
print(h.groupby("system")[["sim_ecapa_cond", "sim_ecapa_eval", "sim_wavlm_eval", "wer_large", "cer_large", "utmos", "latency", "latency_e2e", "rtf"]].mean().round(4))
old = pd.read_csv(os.path.join(EXISTING, "four_system_complete_evaluation.csv")) if os.path.exists(os.path.join(EXISTING, "four_system_complete_evaluation.csv")) else None
print("Consistency with the paper (first 10 sentences, selected cascade, mean ECAPA):",
      round(h[(h.system == "Selected cascade (tau 0.5, 10)") & (h["sample"] <= 10)].sim_ecapa_cond.mean(), 4), "(paper 0.7203)")

In [ ]:
# Cell 10 — TWO EXTRA TARGET REFERENCES (single run each; scored against the reference used and the independent clips)
csv = os.path.join(OUT, "E10_extra_refs.csv"); done = done_keys(csv, ["reference", "system", "sample"])
d10 = os.path.join(OUT, "E10_audio"); os.makedirs(d10, exist_ok=True)
for ref in EXTRA_REFS:
    rn = os.path.basename(ref)
    for name, kind, tau, steps in [("Seed-VC (10 steps)", "sv", None, 10), ("Reference cascade (tau 0.3, 20)", "casc", 0.3, 20),
                                   ("Selected cascade (tau 0.5, 10)", "casc", 0.5, 10)]:
        for _, r in H40.iterrows():
            i = int(r["sample"])
            if (rn, name, str(i)) in done: continue
            out = os.path.join(d10, f"{rn[:-4]}_{re.sub(r'[^A-Za-z0-9]+', '_', name)}_s{i:02d}.wav")
            if kind == "sv": run_seedvc(r.prepared_wav, ref, steps, out, 202900 + i)
            else:
                inter = out.replace(".wav", "_ov.wav"); run_openvoice(r.prepared_wav, ref, tau, inter, 202700 + i); run_seedvc(inter, ref, steps, out, 202900 + i)
            append_row(csv, dict(reference=rn, system=name, sample=i, **score(out, r.transcript, r.prepared_wav, ref, EVAL_CLIPS)))
print(pd.read_csv(csv).groupby(["reference", "system"])[["sim_ecapa_cond", "sim_ecapa_eval", "wer_large", "utmos"]].mean().round(4))

In [ ]:
e10 = pd.read_csv(os.path.join(OUT, "E10_extra_refs.csv"))

print(
    e10.drop_duplicates(
        ["reference", "system", "sample"]
    ).groupby(["reference", "system"]).size()
)

In [ ]:
# Cell 12 — VCTK: 3 speaker pairs, 10 sentences, 5 systems

import soundfile as sf

VCTK = "/content/VCTK"

if not os.path.isfile(os.path.join(VCTK, "speaker-info.txt")):
    print("Downloading VCTK...", flush=True)

    subprocess.run(
        [
            "wget",
            "--progress=dot:giga",
            "-O", "/content/vctk.zip",
            "https://datashare.ed.ac.uk/bitstream/handle/10283/3443/VCTK-Corpus-0.92.zip",
        ],
        check=True,
    )

    print("Extracting VCTK...", flush=True)

    subprocess.run(
        ["unzip", "-q", "/content/vctk.zip", "-d", VCTK],
        check=True,
    )

    os.remove("/content/vctk.zip")

info = {}

with open(
    os.path.join(VCTK, "speaker-info.txt"),
    encoding="utf-8",
) as file:
    for line in file.read().splitlines()[1:]:
        parts = line.split()

        if len(parts) >= 4:
            speaker = (
                parts[0]
                if parts[0].startswith("p")
                else "p" + parts[0]
            )

            info[speaker] = {
                "age": parts[1],
                "gender": parts[2],
                "accent": parts[3],
            }

PAIRS = [
    ("p226", "p225"),
    ("p225", "p226"),
    ("p227", "p232"),
]

for source, target in PAIRS:
    print(
        source, info.get(source),
        "->",
        target, info.get(target),
        flush=True,
    )

WAVDIR = "/content/vctk_wav"
os.makedirs(WAVDIR, exist_ok=True)


def vctk_utts(speaker, minimum_duration, maximum_duration, start, count):
    utterances = []

    pattern = os.path.join(
        VCTK,
        "wav48_silence_trimmed",
        speaker,
        f"{speaker}_*_mic1.flac",
    )

    for audio_path in sorted(glob.glob(pattern)):
        utterance_id = os.path.basename(audio_path).split("_")[1]

        # Skip shared elicitation texts.
        if int(utterance_id) < 25:
            continue

        wav_path = os.path.join(
            WAVDIR,
            f"{speaker}_{utterance_id}.wav",
        )

        if not os.path.exists(wav_path):
            audio, _ = librosa.load(
                audio_path,
                sr=22050,
                mono=True,
            )

            sf.write(
                wav_path,
                audio,
                22050,
                subtype="PCM_16",
            )

        duration = librosa.get_duration(path=wav_path)

        if minimum_duration <= duration <= maximum_duration:
            transcript_path = os.path.join(
                VCTK,
                "txt",
                speaker,
                f"{speaker}_{utterance_id}.txt",
            )

            with open(transcript_path, encoding="utf-8") as file:
                transcript = file.read().strip()

            utterances.append((wav_path, transcript))

    return utterances[start:start + count]


csv = os.path.join(OUT, "E12_vctk_pairs.csv")
done = done_keys(csv, ["pair", "system", "sample"])

acsv = os.path.join(OUT, "E12_vctk_anchors.csv")

d12 = os.path.join(OUT, "E12_audio")
os.makedirs(d12, exist_ok=True)

PAIR_SYSTEMS = [
    ("OpenVoice V2 (tau 0.5)", "ov", 0.5, None),
    ("Seed-VC (10 steps)", "sv", None, 10),
    ("Seed-VC (25 steps, default)", "sv", None, 25),
    ("Reference cascade (tau 0.3, 20)", "casc", 0.3, 20),
    ("Selected cascade (tau 0.5, 10)", "casc", 0.5, 10),
]

for source_speaker, target_speaker in PAIRS:
    pair = f"{source_speaker}->{target_speaker}"

    print(f"\nPreparing {pair}...", flush=True)

    srcs = vctk_utts(
        source_speaker, 3.0, 7.5, 0, 10
    )

    # p232 has no recordings reaching 5.5 seconds.
    # Use the identified 4.952-second recording as its reference.
    if target_speaker == "p232":
        reference_candidates = [
            item
            for item in vctk_utts(
                target_speaker, 2.0, 9.0, 0, 1000
            )
            if os.path.basename(item[0]) == "p232_409.wav"
        ]
    else:
        reference_candidates = vctk_utts(
            target_speaker, 5.5, 9.0, 0, 1
        )

    assert reference_candidates, (
        f"No suitable conditioning reference for {pair}"
    )

    ref = reference_candidates[0][0]

    evals = [
        wav_path
        for wav_path, _ in vctk_utts(
            target_speaker, 2.0, 9.0, 40, 5
        )
    ]

    assert len(srcs) == 10, (
        f"Expected 10 source clips for {pair}, found {len(srcs)}"
    )

    assert len(evals) == 5, (
        f"Expected 5 evaluation clips for {pair}, found {len(evals)}"
    )

    assert ref not in evals, (
        f"Conditioning reference overlaps evaluation clips for {pair}"
    )

    print(
        f"{pair}: {len(srcs)} source clips, "
        f"{len(evals)} evaluation clips",
        flush=True,
    )

    print(
        f"Reference: {os.path.basename(ref)} | "
        f"Duration: {librosa.get_duration(path=ref):.3f} seconds",
        flush=True,
    )

    # Save the reference selection for reporting to the doctor.
    reference_record = {
        "pair": pair,
        "reference": os.path.basename(ref),
        "reference_duration_seconds": float(
            librosa.get_duration(path=ref)
        ),
        "selection": (
            "p232_409 selected because no p232 recording "
            "reaches the original 5.5-second minimum"
            if target_speaker == "p232"
            else "First recording meeting the original 5.5–9.0-second rule"
        ),
        "evaluation_clips": [
            os.path.basename(path) for path in evals
        ],
    }

    with open(
        os.path.join(
            OUT,
            f"E12_reference_{source_speaker}_{target_speaker}.json",
        ),
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(reference_record, file, indent=2)

    if (
        not os.path.exists(acsv)
        or pair not in set(pd.read_csv(acsv)["pair"])
    ):
        for clip in evals:
            append_row(
                acsv,
                dict(
                    pair=pair,
                    anchor="target-target",
                    sim_ecapa=cos(clip, ref, "ecapa"),
                    sim_wavlm=cos(clip, ref, "wavlm"),
                ),
            )

        for source_path, _ in srcs:
            append_row(
                acsv,
                dict(
                    pair=pair,
                    anchor="source-target",
                    sim_ecapa=cos(source_path, ref, "ecapa"),
                    sim_wavlm=cos(source_path, ref, "wavlm"),
                ),
            )

    for name, kind, tau, steps in PAIR_SYSTEMS:
        for sample, (source_path, text) in enumerate(srcs, 1):
            if (pair, name, str(sample)) in done:
                continue

            print(
                f"{pair} | {name} | Sample {sample}/10",
                flush=True,
            )

            tag = re.sub(r"[^A-Za-z0-9]+", "_", name)

            output_path = os.path.join(
                d12,
                f"{source_speaker}_{target_speaker}_"
                f"{tag}_s{sample:02d}.wav",
            )

            ov_seed = 500000 + sample
            sv_seed = 510000 + sample

            if kind == "ov":
                latency = run_openvoice(
                    source_path,
                    ref,
                    tau,
                    output_path,
                    ov_seed,
                )
            elif kind == "sv":
                latency = run_seedvc(
                    source_path,
                    ref,
                    steps,
                    output_path,
                    sv_seed,
                )
            else:
                latency = run_cascade_e2e(
                    source_path,
                    ref,
                    tau,
                    steps,
                    output_path,
                    ov_seed,
                    sv_seed,
                )

            append_row(
                csv,
                dict(
                    pair=pair,
                    source_gender=info.get(
                        source_speaker, {}
                    ).get("gender"),
                    target_gender=info.get(
                        target_speaker, {}
                    ).get("gender"),
                    system=name,
                    sample=sample,
                    reference=os.path.basename(ref),
                    latency=latency,
                    rtf=latency / librosa.get_duration(
                        path=source_path
                    ),
                    **score(
                        output_path,
                        text,
                        source_path,
                        ref,
                        evals,
                    ),
                ),
            )

v = pd.read_csv(csv)

counts = v.drop_duplicates(
    ["pair", "system", "sample"]
).groupby(["pair", "system"]).size()

print("\nSaved results per pair and system:")
print(counts)

assert len(counts) == 15 and counts.eq(10).all(), (
    "VCTK results are incomplete — check the counts above"
)

print(
    v.groupby(["pair", "system"])[
        [
            "sim_ecapa_cond",
            "sim_ecapa_eval",
            "wer_large",
            "utmos",
            "latency",
        ]
    ].mean().round(4)
)

print("✅ VCTK experiments completed — 150 results")

In [ ]:
# Cell 13 — PACK RESULTS (CSVs + environment) AND DOWNLOAD
import zipfile
from google.colab import files
zp = "/content/IJEEE_revision_results_final.zip"
with zipfile.ZipFile(zp, "w") as z:
    for f in glob.glob(os.path.join(OUT, "*.csv")) + glob.glob(os.path.join(OUT, "*.json")) + [HOLDOUT40_META]:
        z.write(f, os.path.basename(f))
print("Files in zip:", len(zipfile.ZipFile(zp).namelist()))
files.download(zp)